# 26 — Silver Pilot Site/Task Seed

## Objetivo

Poblar de forma reproducible los registros del piloto Silver con:

- 12 sitios educativos web,
- 2 tareas por sitio,
- 24 tareas totales.

Después de ejecutar este notebook se debe volver a ejecutar:

`25_silver_pilot_preflight.ipynb`

para comprobar que:

- `registries_ready = True`
- `manifest_rows = 48`
- `pilot_status = READY_FOR_COLLECTION`

Los sitios se proponen como candidatos públicos o de acceso abierto. Antes de iniciar cada captura real, debe verificarse manualmente que el sitio siga accesible y que la tarea pueda realizarse sin utilizar datos personales reales.


## 1. Configuración

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import date

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

META_DIR = (
    PROJECT_DIR
    / "data"
    / "silver"
    / "metadata"
)

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

META_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SITE_PATH = (
    META_DIR
    / "site_registry_pilot.csv"
)

TASK_PATH = (
    META_DIR
    / "task_registry_pilot.csv"
)

VERIFICATION_DATE = "2026-09-07"

print("SITE_PATH:", SITE_PATH)
print("TASK_PATH:", TASK_PATH)


SITE_PATH: /Users/uriel/Documents/machine_learning/data/silver/metadata/site_registry_pilot.csv
TASK_PATH: /Users/uriel/Documents/machine_learning/data/silver/metadata/task_registry_pilot.csv


## 2. Protección contra sobrescritura accidental

El notebook solo reemplaza los registros cuando detecta que las plantillas existentes siguen esencialmente vacías. Si ya fueron editadas manualmente, se detiene.


In [2]:
def registry_has_manual_content(
    path,
    text_columns
):

    if not path.exists():
        return False

    df = pd.read_csv(
        path
    )

    for col in text_columns:

        if col not in df.columns:
            continue

        values = (
            df[
                col
            ]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        if (
            values
            !=
            ""
        ).any():
            return True

    return False


site_manual_content = (
    registry_has_manual_content(
        SITE_PATH,
        [
            "site_name",
            "base_url",
            "domain",
            "educational_context"
        ]
    )
)

task_manual_content = (
    registry_has_manual_content(
        TASK_PATH,
        [
            "task_name",
            "task_description",
            "start_condition",
            "success_criterion"
        ]
    )
)

print(
    "Contenido manual en sites:",
    site_manual_content
)

print(
    "Contenido manual en tasks:",
    task_manual_content
)

if (
    site_manual_content
    or
    task_manual_content
):
    raise RuntimeError(
        "Los registros ya contienen información manual. "
        "No se sobrescribieron. Revisa los CSV antes de continuar."
    )


Contenido manual en sites: False
Contenido manual en tasks: False


## 3. Sitios propuestos

In [3]:
site_rows = [
    {
        "site_id": "site_001",
        "site_name": "PhET Interactive Simulations",
        "base_url": "https://phet.colorado.edu/",
        "domain": "phet.colorado.edu",
        "educational_context": "Simulaciones interactivas de ciencias y matemáticas",
        "access_type": "PUBLIC",
        "requires_account": False,
        "uses_personal_data": False,
        "site_version_or_date": VERIFICATION_DATE,
        "included": True,
        "exclusion_reason": "",
        "notes": "Sitio público. Verificar disponibilidad del simulador seleccionado inmediatamente antes de la captura."
    },
    {
        "site_id": "site_002",
        "site_name": "GeoGebra Calculator Suite",
        "base_url": "https://www.geogebra.org/calculator",
        "domain": "geogebra.org",
        "educational_context": "Matemáticas dinámicas: álgebra, geometría, cálculo y gráficas",
        "access_type": "PUBLIC",
        "requires_account": False,
        "uses_personal_data": False,
        "site_version_or_date": VERIFICATION_DATE,
        "included": True,
        "exclusion_reason": "",
        "notes": "La calculadora web puede utilizarse sin guardar contenido. No iniciar sesión durante el piloto."
    },
    {
        "site_id": "site_003",
        "site_name": "Desmos Graphing Calculator",
        "base_url": "https://www.desmos.com/calculator",
        "domain": "desmos.com",
        "educational_context": "Calculadora gráfica interactiva",
        "access_type": "PUBLIC",
        "requires_account": False,
        "uses_personal_data": False,
        "site_version_or_date": VERIFICATION_DATE,
        "included": True,
        "exclusion_reason": "",
        "notes": "Utilizar únicamente funciones locales de la calculadora; no guardar ni compartir."
    },
    {
        "site_id": "site_004",
        "site_name": "Scratch Online Editor",
        "base_url": "https://scratch.mit.edu/projects/editor/",
        "domain": "scratch.mit.edu",
        "educational_context": "Programación visual educativa",
        "access_type": "PUBLIC",
        "requires_account": False,
        "uses_personal_data": False,
        "site_version_or_date": VERIFICATION_DATE,
        "included": True,
        "exclusion_reason": "",
        "notes": "Usar un proyecto temporal sin iniciar sesión ni publicar."
    },
    {
        "site_id": "site_005",
        "site_name": "Polypad",
        "base_url": "https://mathigon.org/polypad",
        "domain": "mathigon.org",
        "educational_context": "Manipulativos virtuales para matemáticas",
        "access_type": "PUBLIC",
        "requires_account": False,
        "uses_personal_data": False,
        "site_version_or_date": VERIFICATION_DATE,
        "included": True,
        "exclusion_reason": "",
        "notes": "Actualmente redirige al entorno Polypad de Amplify. No guardar ni compartir actividades."
    },
    {
        "site_id": "site_006",
        "site_name": "Numbas",
        "base_url": "https://numbas.mathcentre.ac.uk/",
        "domain": "numbas.mathcentre.ac.uk",
        "educational_context": "Sistema de evaluación matemática web",
        "access_type": "PUBLIC",
        "requires_account": False,
        "uses_personal_data": False,
        "site_version_or_date": VERIFICATION_DATE,
        "included": True,
        "exclusion_reason": "",
        "notes": "Usar exclusivamente la base de datos pública y previews de exámenes; no crear cuenta."
    },
    {
        "site_id": "site_007",
        "site_name": "H5P Examples",
        "base_url": "https://h5p.org/content-types-and-applications",
        "domain": "h5p.org",
        "educational_context": "Ejemplos de contenido educativo interactivo H5P",
        "access_type": "PUBLIC",
        "requires_account": False,
        "uses_personal_data": False,
        "site_version_or_date": VERIFICATION_DATE,
        "included": True,
        "exclusion_reason": "",
        "notes": "Usar previews públicos de tipos de contenido; no crear contenido ni cuenta."
    },
    {
        "site_id": "site_008",
        "site_name": "MDN Playground",
        "base_url": "https://developer.mozilla.org/en-US/play",
        "domain": "developer.mozilla.org",
        "educational_context": "Aprendizaje y experimentación con HTML, CSS y JavaScript",
        "access_type": "PUBLIC",
        "requires_account": False,
        "uses_personal_data": False,
        "site_version_or_date": VERIFICATION_DATE,
        "included": True,
        "exclusion_reason": "",
        "notes": "Trabajar con código mínimo local en el playground; no iniciar sesión."
    },
    {
        "site_id": "site_009",
        "site_name": "W3Schools Tryit Editor",
        "base_url": "https://www.w3schools.com/tryit/",
        "domain": "w3schools.com",
        "educational_context": "Editor y compiladores educativos en navegador",
        "access_type": "PUBLIC",
        "requires_account": False,
        "uses_personal_data": False,
        "site_version_or_date": VERIFICATION_DATE,
        "included": True,
        "exclusion_reason": "",
        "notes": "El inicio de sesión para seguimiento no es necesario para las tareas propuestas."
    },
    {
        "site_id": "site_010",
        "site_name": "freeCodeCamp Learn",
        "base_url": "https://www.freecodecamp.org/learn",
        "domain": "freecodecamp.org",
        "educational_context": "Currículo interactivo de programación web",
        "access_type": "PUBLIC",
        "requires_account": False,
        "uses_personal_data": False,
        "site_version_or_date": VERIFICATION_DATE,
        "included": True,
        "exclusion_reason": "",
        "notes": "Usar lecciones públicas sin crear cuenta. Si una tarea exige login, marcar el sitio/tarea como excluido y sustituirlo."
    },
    {
        "site_id": "site_011",
        "site_name": "OpenStax",
        "base_url": "https://openstax.org/subjects",
        "domain": "openstax.org",
        "educational_context": "Libros de texto abiertos y navegación académica",
        "access_type": "PUBLIC",
        "requires_account": False,
        "uses_personal_data": False,
        "site_version_or_date": VERIFICATION_DATE,
        "included": True,
        "exclusion_reason": "",
        "notes": "Usar lectura/navegación pública de libros; no descargar ni guardar progreso."
    },
    {
        "site_id": "site_012",
        "site_name": "Khan Academy Mathematics",
        "base_url": "https://es.khanacademy.org/math",
        "domain": "es.khanacademy.org",
        "educational_context": "Cursos públicos de matemáticas",
        "access_type": "PUBLIC",
        "requires_account": False,
        "uses_personal_data": False,
        "site_version_or_date": VERIFICATION_DATE,
        "included": True,
        "exclusion_reason": "",
        "notes": "Usar navegación y contenido público. Si una práctica concreta exige sesión, no usar esa práctica."
    }
]

site_registry_seed = pd.DataFrame(
    site_rows
)

print(
    "Sitios:",
    site_registry_seed.shape
)

display(
    site_registry_seed
)


Sitios: (12, 12)


,site_id,site_name,base_url,domain,educational_context,access_type,requires_account,uses_personal_data,site_version_or_date,included,exclusion_reason,notes
0,site_001,PhET Interactive Simulations,https://phet.colorado.edu/,phet.colorado.edu,Simulaciones interactivas de ciencias y matemáticas,PUBLIC,False,False,2026-09-07,True,,Sitio público. Verificar disponibilidad del simulador seleccionado inmediatamente antes de la captura.
1,site_002,GeoGebra Calculator Suite,https://www.geogebra.org/calculator,geogebra.org,"Matemáticas dinámicas: álgebra, geometría, cálculo y gráficas",PUBLIC,False,False,2026-09-07,True,,La calculadora web puede utilizarse sin guardar contenido. No iniciar sesión durante el piloto.
2,site_003,Desmos Graphing Calculator,https://www.desmos.com/calculator,desmos.com,Calculadora gráfica interactiva,PUBLIC,False,False,2026-09-07,True,,Utilizar únicamente funciones locales de la calculadora; no guardar ni compartir.
3,site_004,Scratch Online Editor,https://scratch.mit.edu/projects/editor/,scratch.mit.edu,Programación visual educativa,PUBLIC,False,False,2026-09-07,True,,Usar un proyecto temporal sin iniciar sesión ni publicar.
4,site_005,Polypad,https://mathigon.org/polypad,mathigon.org,Manipulativos virtuales para matemáticas,PUBLIC,False,False,2026-09-07,True,,Actualmente redirige al entorno Polypad de Amplify. No guardar ni compartir actividades.
5,site_006,Numbas,https://numbas.mathcentre.ac.uk/,numbas.mathcentre.ac.uk,Sistema de evaluación matemática web,PUBLIC,False,False,2026-09-07,True,,Usar exclusivamente la base de datos pública y previews de exámenes; no crear cuenta.
6,site_007,H5P Examples,https://h5p.org/content-types-and-applications,h5p.org,Ejemplos de contenido educativo interactivo H5P,PUBLIC,False,False,2026-09-07,True,,Usar previews públicos de tipos de contenido; no crear contenido ni cuenta.
7,site_008,MDN Playground,https://developer.mozilla.org/en-US/play,developer.mozilla.org,"Aprendizaje y experimentación con HTML, CSS y JavaScript",PUBLIC,False,False,2026-09-07,True,,Trabajar con código mínimo local en el playground; no iniciar sesión.
8,site_009,W3Schools Tryit Editor,https://www.w3schools.com/tryit/,w3schools.com,Editor y compiladores educativos en navegador,PUBLIC,False,False,2026-09-07,True,,El inicio de sesión para seguimiento no es necesario para las tareas propuestas.
9,site_010,freeCodeCamp Learn,https://www.freecodecamp.org/learn,freecodecamp.org,Currículo interactivo de programación web,PUBLIC,False,False,2026-09-07,True,,"Usar lecciones públicas sin crear cuenta. Si una tarea exige login, marcar el sitio/tarea como excluido y sustituirlo."


## 4. Tareas propuestas

In [4]:
task_rows = [
    # 001 PhET
    {
        "task_id": "site_001__task_01",
        "site_id": "site_001",
        "task_name": "Abrir una simulación de física",
        "task_description": "Desde la página principal, entrar a Física y abrir una simulación interactiva disponible.",
        "start_condition": "Página principal de PhET cargada.",
        "success_criterion": "La simulación elegida queda abierta y lista para interactuar.",
        "failure_criterion": "No se logra abrir ninguna simulación pública.",
        "expected_page_type": "simulation_catalog_to_simulation",
        "expected_interaction_type": "navigation",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": False,
        "applicable_H6": True,
        "included": True,
        "notes": "Usar la misma simulación en ambas ejecuciones."
    },
    {
        "task_id": "site_001__task_02",
        "site_id": "site_001",
        "task_name": "Modificar un parámetro de simulación",
        "task_description": "Dentro de la simulación seleccionada, cambiar un control visible y observar la respuesta del sistema.",
        "start_condition": "Simulación cargada en su estado inicial.",
        "success_criterion": "Se modifica un parámetro y se observa un cambio visible en la simulación.",
        "failure_criterion": "La simulación no responde o el control no puede utilizarse.",
        "expected_page_type": "interactive_simulation",
        "expected_interaction_type": "control_manipulation",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": False,
        "applicable_H6": True,
        "included": True,
        "notes": "Registrar exactamente qué control se usa."
    },

    # 002 GeoGebra
    {
        "task_id": "site_002__task_01",
        "site_id": "site_002",
        "task_name": "Graficar y=x^2",
        "task_description": "Introducir la expresión y=x^2 en la Suite Calculadora y visualizar la gráfica.",
        "start_condition": "GeoGebra Calculator Suite abierta sin expresiones previas.",
        "success_criterion": "La función y=x^2 aparece representada en la vista gráfica.",
        "failure_criterion": "La expresión no puede introducirse o la gráfica no aparece.",
        "expected_page_type": "graphing_calculator",
        "expected_interaction_type": "text_input_and_render",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": True,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },
    {
        "task_id": "site_002__task_02",
        "site_id": "site_002",
        "task_name": "Cambiar de vista y volver",
        "task_description": "Con la función ya creada, abrir una vista/herramienta secundaria disponible y regresar a la vista gráfica.",
        "start_condition": "Función y=x^2 visible.",
        "success_criterion": "Se cambia de vista y se regresa conservando la expresión o estado esperado.",
        "failure_criterion": "No se puede cambiar de vista o se pierde el estado inesperadamente.",
        "expected_page_type": "multi_view_math_app",
        "expected_interaction_type": "navigation_with_state",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": False,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },

    # 003 Desmos
    {
        "task_id": "site_003__task_01",
        "site_id": "site_003",
        "task_name": "Graficar y=x^2",
        "task_description": "Introducir y=x^2 y verificar que aparezca la curva.",
        "start_condition": "Calculadora Desmos abierta y limpia.",
        "success_criterion": "La curva y=x^2 aparece en el plano.",
        "failure_criterion": "La expresión no se acepta o no se visualiza.",
        "expected_page_type": "graphing_calculator",
        "expected_interaction_type": "text_input_and_render",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": True,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },
    {
        "task_id": "site_003__task_02",
        "site_id": "site_003",
        "task_name": "Crear y modificar un deslizador",
        "task_description": "Introducir y=a*x, crear el deslizador de a y modificar su valor.",
        "start_condition": "Calculadora limpia o con una expresión que pueda eliminarse.",
        "success_criterion": "El deslizador existe y el cambio de a modifica visualmente la gráfica.",
        "failure_criterion": "No se crea el deslizador o el cambio no produce feedback visible.",
        "expected_page_type": "graphing_calculator",
        "expected_interaction_type": "parameter_manipulation",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": True,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },

    # 004 Scratch
    {
        "task_id": "site_004__task_01",
        "site_id": "site_004",
        "task_name": "Agregar y ejecutar un bloque de movimiento",
        "task_description": "Añadir al sprite un bloque 'mover 10 pasos' y ejecutarlo mediante interacción directa.",
        "start_condition": "Editor Scratch abierto con proyecto temporal nuevo.",
        "success_criterion": "El sprite cambia de posición tras ejecutar el bloque.",
        "failure_criterion": "No se puede agregar o ejecutar el bloque.",
        "expected_page_type": "visual_programming_editor",
        "expected_interaction_type": "drag_drop_and_execute",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": False,
        "applicable_H6": True,
        "included": True,
        "notes": "No guardar ni publicar."
    },
    {
        "task_id": "site_004__task_02",
        "site_id": "site_004",
        "task_name": "Crear secuencia de dos bloques",
        "task_description": "Agregar dos bloques compatibles y ejecutar la secuencia observando el resultado.",
        "start_condition": "Proyecto temporal con sprite disponible.",
        "success_criterion": "Los dos bloques quedan conectados y la secuencia produce un resultado visible.",
        "failure_criterion": "Los bloques no pueden conectarse o no se observa resultado.",
        "expected_page_type": "visual_programming_editor",
        "expected_interaction_type": "drag_drop_sequence",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": True,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },

    # 005 Polypad
    {
        "task_id": "site_005__task_01",
        "site_id": "site_005",
        "task_name": "Agregar manipulativos matemáticos",
        "task_description": "Crear un Polypad nuevo y añadir al menos dos objetos matemáticos visibles.",
        "start_condition": "Polypad abierto en un tablero nuevo.",
        "success_criterion": "Dos objetos quedan colocados en el tablero.",
        "failure_criterion": "No se pueden añadir objetos.",
        "expected_page_type": "virtual_manipulative_canvas",
        "expected_interaction_type": "drag_drop",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": False,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },
    {
        "task_id": "site_005__task_02",
        "site_id": "site_005",
        "task_name": "Modificar y eliminar un objeto",
        "task_description": "Seleccionar uno de los objetos, modificar una propiedad disponible y después eliminarlo.",
        "start_condition": "Tablero con al menos dos objetos.",
        "success_criterion": "La modificación se refleja y el objeto puede eliminarse.",
        "failure_criterion": "No se identifica el objeto o no es posible modificar/eliminar.",
        "expected_page_type": "virtual_manipulative_canvas",
        "expected_interaction_type": "object_editing",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": True,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },

    # 006 Numbas
    {
        "task_id": "site_006__task_01",
        "site_id": "site_006",
        "task_name": "Abrir un examen público",
        "task_description": "Desde la página de Numbas, entrar a la base pública y abrir un examen o recurso que disponga de vista previa.",
        "start_condition": "Página inicial de Numbas.",
        "success_criterion": "Se abre un recurso público con una pregunta evaluable.",
        "failure_criterion": "El recurso requiere cuenta o no existe preview.",
        "expected_page_type": "assessment_catalog_to_preview",
        "expected_interaction_type": "navigation",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": False,
        "applicable_H6": True,
        "included": True,
        "notes": "Fijar el mismo recurso antes de iniciar las dos ejecuciones."
    },
    {
        "task_id": "site_006__task_02",
        "site_id": "site_006",
        "task_name": "Responder una pregunta del preview",
        "task_description": "Introducir una respuesta en la primera pregunta disponible y usar el mecanismo de comprobación/submit.",
        "start_condition": "Preview público con pregunta abierta.",
        "success_criterion": "La interfaz procesa la respuesta y muestra feedback o estado evaluativo.",
        "failure_criterion": "La pregunta no puede responderse sin cuenta.",
        "expected_page_type": "web_assessment",
        "expected_interaction_type": "answer_and_feedback",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": True,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },

    # 007 H5P
    {
        "task_id": "site_007__task_01",
        "site_id": "site_007",
        "task_name": "Abrir un ejemplo Course Presentation",
        "task_description": "Localizar Course Presentation entre los ejemplos y abrir su preview público.",
        "start_condition": "Página de ejemplos H5P.",
        "success_criterion": "El ejemplo interactivo queda visible.",
        "failure_criterion": "El ejemplo no está disponible.",
        "expected_page_type": "interactive_content_catalog",
        "expected_interaction_type": "navigation",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": False,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },
    {
        "task_id": "site_007__task_02",
        "site_id": "site_007",
        "task_name": "Navegar e interactuar con un ejemplo H5P",
        "task_description": "Dentro del ejemplo seleccionado, avanzar al menos dos estados/diapositivas y usar una interacción disponible.",
        "start_condition": "Preview H5P cargado.",
        "success_criterion": "Se navega entre estados y se obtiene feedback visible de una interacción.",
        "failure_criterion": "El contenido no permite interacción pública.",
        "expected_page_type": "interactive_learning_object",
        "expected_interaction_type": "navigation_and_response",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": True,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },

    # 008 MDN
    {
        "task_id": "site_008__task_01",
        "site_id": "site_008",
        "task_name": "Crear un ejemplo HTML mínimo",
        "task_description": "En MDN Playground, introducir un encabezado y un párrafo HTML y comprobar el resultado.",
        "start_condition": "Playground abierto.",
        "success_criterion": "La salida renderizada muestra el encabezado y el párrafo.",
        "failure_criterion": "El código no puede editarse o no aparece salida.",
        "expected_page_type": "code_playground",
        "expected_interaction_type": "code_edit_and_preview",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": True,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },
    {
        "task_id": "site_008__task_02",
        "site_id": "site_008",
        "task_name": "Aplicar un cambio CSS",
        "task_description": "Agregar una regla CSS sencilla al contenido creado y observar el cambio visual.",
        "start_condition": "Ejemplo HTML mínimo funcionando.",
        "success_criterion": "La vista previa refleja el cambio CSS.",
        "failure_criterion": "El cambio no puede introducirse o no se refleja.",
        "expected_page_type": "code_playground",
        "expected_interaction_type": "code_edit_and_preview",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": True,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },

    # 009 W3Schools
    {
        "task_id": "site_009__task_01",
        "site_id": "site_009",
        "task_name": "Editar y ejecutar HTML",
        "task_description": "Abrir un editor HTML Tryit, cambiar el texto de un encabezado y ejecutar el código.",
        "start_condition": "Tryit Editor abierto con ejemplo HTML.",
        "success_criterion": "La vista de resultado muestra el nuevo texto.",
        "failure_criterion": "No se puede editar o ejecutar el ejemplo.",
        "expected_page_type": "online_code_editor",
        "expected_interaction_type": "code_edit_and_run",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": True,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },
    {
        "task_id": "site_009__task_02",
        "site_id": "site_009",
        "task_name": "Modificar estilo CSS y ejecutar",
        "task_description": "Modificar una propiedad CSS visible y ejecutar nuevamente el ejemplo.",
        "start_condition": "Editor con HTML funcional.",
        "success_criterion": "La salida cambia según la propiedad CSS.",
        "failure_criterion": "No se refleja el cambio.",
        "expected_page_type": "online_code_editor",
        "expected_interaction_type": "code_edit_and_run",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": True,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },

    # 010 freeCodeCamp
    {
        "task_id": "site_010__task_01",
        "site_id": "site_010",
        "task_name": "Abrir una lección pública de HTML/CSS",
        "task_description": "Desde el currículo, entrar a una lección pública inicial de desarrollo web.",
        "start_condition": "Página Learn de freeCodeCamp.",
        "success_criterion": "La lección y su editor/instrucciones quedan visibles sin iniciar sesión.",
        "failure_criterion": "La lección elegida exige autenticación.",
        "expected_page_type": "interactive_lesson",
        "expected_interaction_type": "navigation",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": False,
        "applicable_H6": True,
        "included": True,
        "notes": "Si la lección exige cuenta, sustituir por otra lección pública."
    },
    {
        "task_id": "site_010__task_02",
        "site_id": "site_010",
        "task_name": "Modificar solución y ejecutar pruebas",
        "task_description": "En la lección pública seleccionada, realizar un cambio solicitado y ejecutar el mecanismo de tests/check.",
        "start_condition": "Lección interactiva abierta.",
        "success_criterion": "La plataforma procesa el intento y muestra el resultado de las pruebas.",
        "failure_criterion": "La función requiere autenticación.",
        "expected_page_type": "interactive_lesson",
        "expected_interaction_type": "code_edit_and_test",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": True,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },

    # 011 OpenStax
    {
        "task_id": "site_011__task_01",
        "site_id": "site_011",
        "task_name": "Abrir un libro y una sección",
        "task_description": "Desde Subjects, seleccionar un libro abierto y navegar hasta una sección mediante su tabla de contenidos.",
        "start_condition": "Página Subjects de OpenStax.",
        "success_criterion": "Se abre una sección concreta del libro.",
        "failure_criterion": "El contenido seleccionado no está disponible públicamente.",
        "expected_page_type": "textbook_navigation",
        "expected_interaction_type": "navigation",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": False,
        "applicable_H6": True,
        "included": True,
        "notes": "Fijar el mismo libro y sección antes de ambas ejecuciones."
    },
    {
        "task_id": "site_011__task_02",
        "site_id": "site_011",
        "task_name": "Navegar entre secciones del libro",
        "task_description": "Desde una sección, avanzar a la siguiente sección y regresar usando navegación visible.",
        "start_condition": "Sección de libro abierta.",
        "success_criterion": "Se navega a otra sección y se regresa correctamente.",
        "failure_criterion": "No hay mecanismo de navegación visible o falla.",
        "expected_page_type": "textbook_reader",
        "expected_interaction_type": "navigation_with_backtracking",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": False,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    },

    # 012 Khan
    {
        "task_id": "site_012__task_01",
        "site_id": "site_012",
        "task_name": "Abrir una unidad de matemáticas",
        "task_description": "Desde Matemáticas, elegir un curso y abrir una unidad pública.",
        "start_condition": "Página de Matemáticas de Khan Academy.",
        "success_criterion": "La unidad y sus contenidos quedan visibles.",
        "failure_criterion": "El contenido exige autenticación antes de poder visualizarse.",
        "expected_page_type": "course_navigation",
        "expected_interaction_type": "navigation",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": False,
        "applicable_H6": True,
        "included": True,
        "notes": "No crear cuenta ni guardar progreso."
    },
    {
        "task_id": "site_012__task_02",
        "site_id": "site_012",
        "task_name": "Abrir una lección y volver a la unidad",
        "task_description": "Abrir una lección pública de la unidad y regresar al listado mediante la navegación disponible.",
        "start_condition": "Unidad pública abierta.",
        "success_criterion": "Se abre una lección y se regresa al contexto de la unidad.",
        "failure_criterion": "La navegación requiere autenticación o no funciona.",
        "expected_page_type": "lesson_navigation",
        "expected_interaction_type": "navigation_with_backtracking",
        "applicable_H1": True,
        "applicable_H4": True,
        "applicable_H5": False,
        "applicable_H6": True,
        "included": True,
        "notes": ""
    }
]

task_registry_seed = pd.DataFrame(
    task_rows
)

print(
    "Tareas:",
    task_registry_seed.shape
)

display(
    task_registry_seed
)


Tareas: (24, 15)


,task_id,site_id,task_name,task_description,start_condition,success_criterion,failure_criterion,expected_page_type,expected_interaction_type,applicable_H1,applicable_H4,applicable_H5,applicable_H6,included,notes
0,site_001__task_01,site_001,Abrir una simulación de física,"Desde la página principal, entrar a Física y abrir una simulación interactiva disponible.",Página principal de PhET cargada.,La simulación elegida queda abierta y lista para interactuar.,No se logra abrir ninguna simulación pública.,simulation_catalog_to_simulation,navigation,True,True,False,True,True,Usar la misma simulación en ambas ejecuciones.
1,site_001__task_02,site_001,Modificar un parámetro de simulación,"Dentro de la simulación seleccionada, cambiar un control visible y observar la respuesta del sistema.",Simulación cargada en su estado inicial.,Se modifica un parámetro y se observa un cambio visible en la simulación.,La simulación no responde o el control no puede utilizarse.,interactive_simulation,control_manipulation,True,True,False,True,True,Registrar exactamente qué control se usa.
2,site_002__task_01,site_002,Graficar y=x^2,Introducir la expresión y=x^2 en la Suite Calculadora y visualizar la gráfica.,GeoGebra Calculator Suite abierta sin expresiones previas.,La función y=x^2 aparece representada en la vista gráfica.,La expresión no puede introducirse o la gráfica no aparece.,graphing_calculator,text_input_and_render,True,True,True,True,True,
3,site_002__task_02,site_002,Cambiar de vista y volver,"Con la función ya creada, abrir una vista/herramienta secundaria disponible y regresar a la vista gráfica.",Función y=x^2 visible.,Se cambia de vista y se regresa conservando la expresión o estado esperado.,No se puede cambiar de vista o se pierde el estado inesperadamente.,multi_view_math_app,navigation_with_state,True,True,False,True,True,
4,site_003__task_01,site_003,Graficar y=x^2,Introducir y=x^2 y verificar que aparezca la curva.,Calculadora Desmos abierta y limpia.,La curva y=x^2 aparece en el plano.,La expresión no se acepta o no se visualiza.,graphing_calculator,text_input_and_render,True,True,True,True,True,
5,site_003__task_02,site_003,Crear y modificar un deslizador,"Introducir y=a*x, crear el deslizador de a y modificar su valor.",Calculadora limpia o con una expresión que pueda eliminarse.,El deslizador existe y el cambio de a modifica visualmente la gráfica.,No se crea el deslizador o el cambio no produce feedback visible.,graphing_calculator,parameter_manipulation,True,True,True,True,True,
6,site_004__task_01,site_004,Agregar y ejecutar un bloque de movimiento,Añadir al sprite un bloque 'mover 10 pasos' y ejecutarlo mediante interacción directa.,Editor Scratch abierto con proyecto temporal nuevo.,El sprite cambia de posición tras ejecutar el bloque.,No se puede agregar o ejecutar el bloque.,visual_programming_editor,drag_drop_and_execute,True,True,False,True,True,No guardar ni publicar.
7,site_004__task_02,site_004,Crear secuencia de dos bloques,Agregar dos bloques compatibles y ejecutar la secuencia observando el resultado.,Proyecto temporal con sprite disponible.,Los dos bloques quedan conectados y la secuencia produce un resultado visible.,Los bloques no pueden conectarse o no se observa resultado.,visual_programming_editor,drag_drop_sequence,True,True,True,True,True,
8,site_005__task_01,site_005,Agregar manipulativos matemáticos,Crear un Polypad nuevo y añadir al menos dos objetos matemáticos visibles.,Polypad abierto en un tablero nuevo.,Dos objetos quedan colocados en el tablero.,No se pueden añadir objetos.,virtual_manipulative_canvas,drag_drop,True,True,False,True,True,
9,site_005__task_02,site_005,Modificar y eliminar un objeto,"Seleccionar uno de los objetos, modificar una propiedad disponible y después eliminarlo.",Tablero con al menos dos objetos.,La modificación se refleja y el objeto puede eliminarse.,No se identifica el objeto o no es posible modificar/eliminar.,virtual_manipulative_canvas,object_editing,True,True,True,T

## 5. Validaciones antes de guardar

In [5]:
assert len(
    site_registry_seed
) == 12

assert (
    site_registry_seed[
        "site_id"
    ]
    .duplicated()
    .sum()
    ==
    0
)

assert len(
    task_registry_seed
) == 24

assert (
    task_registry_seed[
        "task_id"
    ]
    .duplicated()
    .sum()
    ==
    0
)

assert set(
    task_registry_seed[
        "site_id"
    ]
).issubset(
    set(
        site_registry_seed[
            "site_id"
        ]
    )
)

tasks_per_site = (
    task_registry_seed
    .groupby(
        "site_id"
    )
    .size()
)

assert (
    tasks_per_site
    ==
    2
).all()

assert (
    site_registry_seed[
        "uses_personal_data"
    ]
    ==
    False
).all()

assert (
    site_registry_seed[
        "included"
    ]
    ==
    True
).all()

assert (
    task_registry_seed[
        "included"
    ]
    ==
    True
).all()

print(
    "12 sitios × 2 tareas = 24 tareas ✅"
)

print(
    "Sin IDs duplicados ✅"
)

print(
    "Sin uso planificado de datos personales reales ✅"
)


12 sitios × 2 tareas = 24 tareas ✅
Sin IDs duplicados ✅
Sin uso planificado de datos personales reales ✅


## 6. Guardar registros

In [6]:
site_registry_seed.to_csv(
    SITE_PATH,
    index=False
)

task_registry_seed.to_csv(
    TASK_PATH,
    index=False
)

print(
    SITE_PATH.name,
    "✅"
    if SITE_PATH.exists()
    else "❌"
)

print(
    TASK_PATH.name,
    "✅"
    if TASK_PATH.exists()
    else "❌"
)


site_registry_pilot.csv ✅
task_registry_pilot.csv ✅


## 7. Guardar auditoría de procedencia del seed

In [7]:
seed_provenance = pd.DataFrame([
    {
        "dataset":
            "OWN_LLM_SILVER_WEB",

        "artifact":
            "site_registry_pilot.csv",

        "seed_date":
            VERIFICATION_DATE,

        "sites":
            12,

        "selection_basis":
            "Public educational web interfaces with reproducible tasks and no required use of personal data.",

        "manual_pre_collection_check_required":
            True
    },

    {
        "dataset":
            "OWN_LLM_SILVER_WEB",

        "artifact":
            "task_registry_pilot.csv",

        "seed_date":
            VERIFICATION_DATE,

        "sites":
            12,

        "selection_basis":
            "Two predefined reproducible tasks per site; applicability flags are opportunities to assess heuristics, not labels.",

        "manual_pre_collection_check_required":
            True
    }
])

seed_provenance.to_csv(
    REPORTS_DIR
    / "silver_pilot_seed_provenance.csv",
    index=False
)

display(
    seed_provenance
)


,dataset,artifact,seed_date,sites,selection_basis,manual_pre_collection_check_required
0,OWN_LLM_SILVER_WEB,site_registry_pilot.csv,2026-09-07,12,Public educational web interfaces with reproducible tasks and no required use of personal data.,True
1,OWN_LLM_SILVER_WEB,task_registry_pilot.csv,2026-09-07,12,"Two predefined reproducible tasks per site; applicability flags are opportunities to assess heuristics, not labels.",True


## Siguiente paso

1. Abrir manualmente cada uno de los 12 sitios una vez y confirmar que las dos tareas siguen siendo realizables sin datos personales reales.
2. Si un sitio/tarea ya no funciona, cambiar `included=False` y documentar `exclusion_reason` antes de sustituirlo.
3. Volver a ejecutar `25_silver_pilot_preflight.ipynb`.

El resultado objetivo después de reejecutar el notebook 25 es:

```text
Registries ready: True
Pilot manifest: (48, 8)

registries_ready = True
manifest_rows = 48
exports_found = 0
valid_exports = 0
pilot_status = READY_FOR_COLLECTION
```

Ese estado significará que ya podemos comenzar la captura real con la extensión.
